# This script is implemented to manage and process instances that encountered Out of Memory (OOM) errors during the initial evaluation phase.

The workflow operates by identifying unassigned or unannotated chunks—which were left unevaluated due to OOM errors—and resubmitting them to the Judge model for scoring. Finally, once it is verified that all chunks have been assigned a score, the evaluation metrics for the Retriever module are calculated.


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!nvidia-smi

In [ ]:
import os
import json
import re
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


INPUT_FILE = "/content/retrieval_baseline_final.json"
OUTPUT_FILE = "/content/relevance_evaluation_final.json"


MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

print("Model loaded successfully.")

In [ ]:
INPUT_FILE = "/content/retrieval_baseline_final.json"
OUTPUT_FILE = "/content/relevance_evaluation_final.json"


## added these filed for "OOM" error
RETRIEVAL_FILE = "/content/retrieval_baseline_final.json"
JUDGE_FILE = "/content/relevance_evaluation_final.json"
OUTPUT_FILE = "/content/relevance_evaluation_full_fixed.json"

In [ ]:
# Because of "OOM" prompt has been changed.
SYSTEM_PROMPT = """
You are an expert evaluator of information retrieval
for a medical question-answering system.

Your task is to evaluate ONE retrieved chunk for its usefulness
in answering the given multiple-choice medical question.

Scoring rubric:

4 = The chunk contains sufficient information to directly
    support the correct answer.

3 = The chunk is strongly relevant and contains evidence
    supporting the correct answer, but additional information
    may be needed.

2 = The chunk is related to the topic but does not provide
    meaningful evidence for the correct answer.

1 = The chunk has only weak or tangential relevance.

0 = The chunk is irrelevant.

Return ONLY a single integer:
0
1
2
3
or
4

Do not provide any explanation.
Do not provide reasoning.
Do not use markdown.
Do not output <think>.
"""

def format_options(options):

    lines = []

    for letter in ["a", "b", "c", "d", "e"]:

        if letter in options:
            lines.append(
                f"{letter}. {options[letter]}"
            )

    return "\n".join(lines)

def format_chunks(retrieved):

    blocks = []

    for item in retrieved:

        block = (
            f"--- Chunk Rank {item['rank']} ---\n"
            f"{item['text']}"
        )

        blocks.append(block)

    return "\n\n".join(blocks)




def build_prompt(item, chunk_text):

    options_text = format_options(
        item.get("options", {})
    )

    prompt = f"""
Question:
{item["question"]}

Options:
{options_text}

Correct answer:
{item["answer"]}

Reference answer:
{item.get("answer_text", "")}

Reference explanation:
{item.get("explanation", "")}

Retrieved chunk:
{chunk_text}

Evaluate the relevance of this ONE retrieved chunk.

Return ONLY one integer from 0 to 4.
"""

    return prompt


def generate_judgment(prompt):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()



def parse_json_response(response):

    response = response.strip()
    response = re.sub(
        r"```json\s*",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = re.sub(
        r"```\s*",
        "",
        response
    )

    response = response.strip()
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass
    match = re.search(
        r"\{.*\}",
        response,
        flags=re.DOTALL
    )

    if match:

        json_text = match.group(0)

        try:
            return json.loads(json_text)

        except json.JSONDecodeError:
            pass

    raise ValueError(
        f"Could not parse model response as JSON:\n{response}"
    )


def judge_chunk(item, chunk_text):
    MAX_CHUNK_CHARS = 6000

    if len(chunk_text) > MAX_CHUNK_CHARS:
        chunk_text = chunk_text[:MAX_CHUNK_CHARS]

    prompt = build_prompt(
        item,
        chunk_text
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            use_cache=True
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[
        0,
        input_length:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    del inputs
    del outputs
    del generated_tokens

    torch.cuda.empty_cache()
    gc.collect()
    match = re.fullmatch(
        r"\s*([0-4])\s*",
        response
    )

    if not match:
        raise ValueError(
            f"Invalid judge output: {response}"
        )

    return int(match.group(1))


def validate_evaluation(
    evaluation,
    retrieved
):

    if "evaluations" not in evaluation:
        raise ValueError(
            "Missing 'evaluations' field."
        )

    evaluations = evaluation["evaluations"]

    expected_ranks = {
        item["rank"]
        for item in retrieved
    }

    returned_ranks = {
        item.get("rank")
        for item in evaluations
    }

    if expected_ranks != returned_ranks:
        raise ValueError(
            f"Rank mismatch. "
            f"Expected {expected_ranks}, "
            f"got {returned_ranks}"
        )

    for item in evaluations:

        score = item.get(
            "relevance_score"
        )

        if not isinstance(score, int):
            raise ValueError(
                f"Invalid score: {score}"
            )

        if score < 0 or score > 4:
            raise ValueError(
                f"Score must be 0-4, got {score}"
            )

    return True

In [ ]:
import json
import time
import gc
import torch

with open(
    RETRIEVAL_FILE,
    "r",
    encoding="utf-8"
) as f:

    retrieval_data = json.load(f)


with open(
    JUDGE_FILE,
    "r",
    encoding="utf-8"
) as f:

    judge_data = json.load(f)


print(
    "Retrieval questions:",
    len(retrieval_data)
)

print(
    "Judge records:",
    len(judge_data)
)


error_items = [
    item
    for item in judge_data
    if "error" in item
]

print(
    "Questions with errors:",
    len(error_items)
)

for item in error_items:

    print(
        item["question_id"],
        "->",
        item.get("error", "")[:80]
    )



retrieval_lookup = {
    item["id"]: item
    for item in retrieval_data
}

print(
    "Retrieval lookup:",
    len(retrieval_lookup)
)

RETRY_CHECKPOINT = "/content/judge_retry_checkpoint.json"


if os.path.exists(RETRY_CHECKPOINT):

    with open(
        RETRY_CHECKPOINT,
        "r",
        encoding="utf-8"
    ) as f:

        retry_results = json.load(f)

else:

    retry_results = {}

def save_retry_checkpoint():

    temp_file = RETRY_CHECKPOINT + ".tmp"

    with open(
        temp_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            retry_results,
            f,
            ensure_ascii=False,
            indent=2
        )

    os.replace(
        temp_file,
        RETRY_CHECKPOINT
    )



    import time

for index, error_item in enumerate(
    error_items,
    start=1
):

    question_id = error_item["question_id"]

    print(
        "\n" + "=" * 60
    )

    print(
        f"[{index}/{len(error_items)}] "
        f"{question_id}"
    )

    # Retrieve original question
    retrieval_item = retrieval_lookup.get(
        question_id
    )

    if retrieval_item is None:

        print(
            "ERROR: Retrieval data not found!"
        )

        continue

    retrieved_chunks = retrieval_item.get(
        "retrieved",
        []
    )

    print(
        f"Found {len(retrieved_chunks)} retrieved chunks."
    )

    if question_id not in retry_results:
        retry_results[question_id] = {}


    for chunk in retrieved_chunks:

        rank = chunk["rank"]

        # Resume support
        if str(rank) in retry_results[question_id]:

            print(
                f"  Rank {rank}: SKIPPED"
            )

            continue

        print(
            f"Rank {rank}: judging...",
            end=" "
        )

        try:

            start_time = time.time()

            score = judge_chunk(
                retrieval_item,
                chunk["text"]
            )

            elapsed = time.time() - start_time

            retry_results[
                question_id
            ][str(rank)] = score

            # SAVE IMMEDIATELY
            save_retry_checkpoint()

            print(
                f"score={score} "
                f"({elapsed:.1f}s)"
            )

        except torch.cuda.OutOfMemoryError:

            print(
                "OOM"
            )

            # Clean memory
            gc.collect()
            torch.cuda.empty_cache()

            # Wait a little
            time.sleep(3)

            continue

        except Exception as e:

            print(
                f"✗ ERROR: {e}"
            )

            continue

fixed_results = []

for item in judge_data:

    question_id = item["question_id"]
    if "error" not in item:

        fixed_results.append(item)

        continue
    scores = retry_results.get(
        question_id,
        {}
    )

    evaluations = []

    for rank in range(1, 11):

        rank_str = str(rank)

        if rank_str not in scores:

            print(
                f"WARNING: "
                f"{question_id} "
                f"missing rank {rank}"
            )

            continue

        evaluations.append(
            {
                "rank": rank,
                "relevance_score": scores[rank_str]
            }
        )

    fixed_item = {
        "question_id":
            question_id,

        "chapter":
            item.get("chapter", ""),

        "question_number":
            item.get("question_number", ""),

        "question":
            item.get("question", ""),

        "correct_answer":
            item.get("correct_answer", ""),

        "answer_text":
            item.get("answer_text", ""),

        "explanation":
            item.get("explanation", ""),

        "evaluations":
            evaluations
    }

    fixed_results.append(
        fixed_item
    )



temp_output = OUTPUT_FILE + ".tmp"

with open(
    temp_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        fixed_results,
        f,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    temp_output,
    OUTPUT_FILE
)

print(
    "Saved:",
    OUTPUT_FILE
)



print("\nValidating final file...\n")

with open(
    OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    final_data = json.load(f)

errors = []
incomplete = []

for item in final_data:

    if "error" in item:

        errors.append(
            item["question_id"]
        )

    evaluations = item.get(
        "evaluations",
        []
    )

    if len(evaluations) != 10:

        incomplete.append(
            (
                item["question_id"],
                len(evaluations)
            )
        )

print(
    "Total questions:",
    len(final_data)
)

print(
    "Remaining errors:",
    len(errors)
)

print(
    "Incomplete questions:",
    len(incomplete)
)

if errors:

    print(
        "\nRemaining errors:",
        errors
    )

if incomplete:

    print(
        "\nIncomplete:",
        incomplete
    )

Retrieval questions: 232
Judge records: 232
Questions with errors: 13
CAMPBELL_0004 -> CUDA out of memory. Tried to allocate 62.50 GiB. GPU 0 has a total capacity of 1
CAMPBELL_0020 -> CUDA out of memory. Tried to allocate 31.22 GiB. GPU 0 has a total capacity of 1
CAMPBELL_0027 -> CUDA out of memory. Tried to allocate 16.61 GiB. GPU 0 has a total capacity of 1
CAMPBELL_0028 -> CUDA out of memory. Tried to allocate 63.27 GiB. GPU 0 has a total capacity of 1
CAMPBELL_0170 -> CUDA out of memory. Tried to allocate 4.02 GiB. GPU 0 has a total capacity of 14
CAMPBELL_0171 -> CUDA out of memory. Tried to allocate 2.11 GiB. GPU 0 has a total capacity of 14
CAMPBELL_0190 -> CUDA out of memory. Tried to allocate 7.60 GiB. GPU 0 has a total capacity of 14
CAMPBELL_0206 -> CUDA out of memory. Tried to allocate 7.20 GiB. GPU 0 has a total capacity of 14
CAMPBELL_0207 -> CUDA out of memory. Tried to allocate 8.90 GiB. GPU 0 has a total capacity of 14
CAMPBELL_0209 -> CUDA out of memory. Tried to al

In [ ]:
missing = {
    "CAMPBELL_0004": 2,
    "CAMPBELL_0020": 10,
    "CAMPBELL_0027": 4,
    "CAMPBELL_0028": 8,
    "CAMPBELL_0190": 1,
    "CAMPBELL_0206": 5,
    "CAMPBELL_0207": 4,
    "CAMPBELL_0221": 10,
}

import time
import gc
import torch

for i, (question_id, target_rank) in enumerate(
    missing.items(),
    start=1
):

    print(
        f"\n[{i}/{len(missing)}] "
        f"{question_id} - Rank {target_rank}"
    )

    retrieval_item = retrieval_lookup.get(question_id)

    if retrieval_item is None:
        print("✗ Retrieval data not found!")
        continue

    target_chunk = None

    for chunk in retrieval_item.get("retrieved", []):

        if chunk["rank"] == target_rank:
            target_chunk = chunk
            break

    if target_chunk is None:
        print("✗ Target chunk not found!")
        continue

    print("Judging...", end=" ")

    try:

        start_time = time.time()

        score = judge_chunk(
            retrieval_item,
            target_chunk["text"]
        )

        elapsed = time.time() - start_time
        if question_id not in retry_results:
            retry_results[question_id] = {}

        retry_results[question_id][
            str(target_rank)
        ] = score

        save_retry_checkpoint()

        print(
            f"✓ score={score} "
            f"({elapsed:.1f}s)"
        )

    except torch.cuda.OutOfMemoryError:

        print("✗ CUDA OOM")

        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:

        print(f"✗ ERROR: {e}")


for qid, rank in missing.items():

    print(
        qid,
        "rank",
        rank,
        "=>",
        retry_results.get(qid, {}).get(
            str(rank),
            "NOT FOUND"
        )
    )


fixed_results = []

for item in judge_data:

    question_id = item["question_id"]

    if "error" not in item:
        fixed_results.append(item)
        continue

    scores = retry_results.get(
        question_id,
        {}
    )

    evaluations = []

    for rank in range(1, 11):

        rank_str = str(rank)

        if rank_str not in scores:
            print(
                f"WARNING: "
                f"{question_id} "
                f"missing rank {rank}"
            )
            continue

        evaluations.append({
            "rank": rank,
            "relevance_score": scores[rank_str]
        })

    fixed_item = {
        "question_id": question_id,
        "chapter": item.get("chapter", ""),
        "question_number": item.get("question_number", ""),
        "question": item.get("question", ""),
        "correct_answer": item.get("correct_answer", ""),
        "answer_text": item.get("answer_text", ""),
        "explanation": item.get("explanation", ""),
        "evaluations": evaluations
    }

    fixed_results.append(fixed_item)


temp_output = OUTPUT_FILE + ".tmp"

with open(
    temp_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        fixed_results,
        f,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    temp_output,
    OUTPUT_FILE
)

print("Saved:", OUTPUT_FILE)

errors = []
incomplete = []

for item in fixed_results:

    if "error" in item:
        errors.append(item["question_id"])

    if len(item.get("evaluations", [])) != 10:
        incomplete.append(
            (
                item["question_id"],
                len(item.get("evaluations", []))
            )
        )

print("Total questions:", len(fixed_results))
print("Remaining errors:", len(errors))
print("Incomplete questions:", len(incomplete))

if incomplete:
    print("\nIncomplete:")
    print(incomplete)


[1/8] CAMPBELL_0004 - Rank 2
Judging... ✓ score=2 (3.3s)

[2/8] CAMPBELL_0020 - Rank 10
Judging... ✓ score=4 (3.5s)

[3/8] CAMPBELL_0027 - Rank 4
Judging... ✓ score=2 (4.2s)

[4/8] CAMPBELL_0028 - Rank 8
Judging... ✓ score=3 (3.6s)

[5/8] CAMPBELL_0190 - Rank 1
Judging... ✓ score=3 (4.4s)

[6/8] CAMPBELL_0206 - Rank 5
Judging... ✓ score=3 (4.5s)

[7/8] CAMPBELL_0207 - Rank 4
Judging... ✓ score=3 (4.6s)

[8/8] CAMPBELL_0221 - Rank 10
Judging... ✓ score=4 (4.6s)
CAMPBELL_0004 rank 2 => 2
CAMPBELL_0020 rank 10 => 4
CAMPBELL_0027 rank 4 => 2
CAMPBELL_0028 rank 8 => 3
CAMPBELL_0190 rank 1 => 3
CAMPBELL_0206 rank 5 => 3
CAMPBELL_0207 rank 4 => 3
CAMPBELL_0221 rank 10 => 4
Saved: /content/relevance_evaluation_full_fixed.json
Total questions: 232
Remaining errors: 0
Incomplete questions: 0


In [ ]:
import json
import numpy as np
import pandas as pd

JUDGE_FILE = "/content/relevance_evaluation_full_fixed.json"

with open(JUDGE_FILE, "r", encoding="utf-8") as f:
    judge_data = json.load(f)

print(f"Loaded {len(judge_data)} questions")

RELEVANCE_THRESHOLD = 3
K_VALUES = [1, 3, 5, 10]

results = {
    "Hit@1": [],
    "Hit@3": [],
    "Hit@5": [],
    "Hit@10": [],
    "MRR": [],
    "Mean Relevance@1": [],
    "Mean Relevance@3": [],
    "Mean Relevance@5": [],
    "Mean Relevance@10": [],
}

for item in judge_data:

    evaluations = sorted(
        item["evaluations"],
        key=lambda x: x["rank"]
    )

    scores = [e["relevance_score"] for e in evaluations]
    for k in K_VALUES:
        top_k = scores[:k]

        hit = int(
            any(score >= RELEVANCE_THRESHOLD for score in top_k)
        )

        results[f"Hit@{k}"].append(hit)

    first_relevant_rank = None

    for rank, score in enumerate(scores, start=1):
        if score >= RELEVANCE_THRESHOLD:
            first_relevant_rank = rank
            break

    if first_relevant_rank is not None:
        reciprocal_rank = 1 / first_relevant_rank
    else:
        reciprocal_rank = 0.0

    results["MRR"].append(reciprocal_rank)

    for k in K_VALUES:
        mean_relevance = np.mean(scores[:k])
        results[f"Mean Relevance@{k}"].append(mean_relevance)

summary = {
    metric: np.mean(values)
    for metric, values in results.items()
}

print("\n" + "=" * 50)
print("RETRIEVAL EVALUATION RESULTS")
print("=" * 50)

for metric, value in summary.items():
    print(f"{metric:25s}: {value:.4f}")

Loaded 232 questions

RETRIEVAL EVALUATION RESULTS
Hit@1                    : 0.7759
Hit@3                    : 0.9655
Hit@5                    : 0.9871
Hit@10                   : 0.9957
MRR                      : 0.8735
Mean Relevance@1         : 3.2974
Mean Relevance@3         : 3.0273
Mean Relevance@5         : 2.8422
Mean Relevance@10        : 2.6815
